# MTAOS Per-Sensor Zernike Coefficients — EFD Time Series

Queries `lsst.sal.MTAOS.logevent_wavefrontError` from the EFD to retrieve the
measured annular Zernike coefficients reported by the AOS wavefront estimation
pipeline (WEP) for each of the four corner wavefront sensors.

Each event is emitted once per corner sensor after a pair of intra/extra-focal
donut images are processed.  Fields:

| EFD field | Noll index | Aberration |
|---|---|---|
| `annularZernikeCoeff0` | Z4 | Defocus |
| `annularZernikeCoeff1` | Z5 | Oblique astigmatism |
| `annularZernikeCoeff2` | Z6 | Vertical astigmatism |
| `annularZernikeCoeff3` | Z7 | Vertical coma |
| `annularZernikeCoeff4` | Z8 | Horizontal coma |
| … | … | … |
| `annularZernikeCoeff18` | Z22 | — |

The `sensorId` field identifies which corner sensor produced the measurement.
Typical LSSTCam corner sensor IDs are mapped in the **Configuration** cell below.

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from astropy.time import Time, TimeDelta
import astropy.units as u

from lsst_efd_client import EfdClient

%matplotlib inline

## Configuration

In [ ]:
# ── Night to query ────────────────────────────────────────────────────────────
# Set to the TAI date of the night (evening date, YYYY-MM-DD).
# The window covers 12:00 UTC on that date to 12:00 UTC the next day,
# which brackets a full Rubin observing night.
NIGHT_DATE = "2025-11-15"   # <-- change this

t_start = Time(f"{NIGHT_DATE}T12:00:00", scale="utc")
t_end   = t_start + TimeDelta(1 * u.day)
print(f"Query window: {t_start.iso}  →  {t_end.iso}  (UTC)")

# ── EFD ───────────────────────────────────────────────────────────────────────
EFD_ALIAS = "usdf_efd"
WFE_TOPIC = "lsst.sal.MTAOS.logevent_wavefrontError"

# EFD stores Zernike coefficients as:
#   nollZernikeValues0  → Z4  (Noll index 4, defocus)
#   nollZernikeValues1  → Z5  (oblique astigmatism)
#   nollZernikeValues2  → Z6  (vertical astigmatism)
#   ...
#   nollZernikeValues24 → Z28
N_NOLL_VALUES = 25   # number of Zernike values stored (Z4–Z28)
NOLL_OFFSET   = 4    # first Noll index

# ── Corner sensor ID → label mapping ─────────────────────────────────────────
SENSOR_LABELS = {
    191: "R00 (SW corner)",
    195: "R04 (SE corner)",
    199: "R40 (NW corner)",
    203: "R44 (NE corner)",
}
SENSOR_COLORS = {
    191: "steelblue",
    195: "tomato",
    199: "mediumseagreen",
    203: "darkorchid",
}


Query window: 2025-11-15 12:00:00.000  →  2025-11-16 12:00:00.000  (UTC)


## Connect to EFD

In [3]:
efd_client = EfdClient(EFD_ALIAS)
print(f"Connected to EFD alias: {EFD_ALIAS}")

Connected to EFD alias: usdf_efd


## Query `logevent_wavefrontError`

In [ ]:
# Fetch all wavefront error events for the night.
# Each row = one sensor per AOS cycle.
# Zernike data is in nollZernikeValues0-24 (Noll Z4-Z28)
# with corresponding nollZernikeIndices0-24 confirming the Noll index.
noll_val_fields = [f"nollZernikeValues{i}" for i in range(N_NOLL_VALUES)]
noll_idx_fields = [f"nollZernikeIndices{i}" for i in range(N_NOLL_VALUES)]

df_wfe = await efd_client.select_time_series(
    WFE_TOPIC,
    fields=["sensorId", "visitId"] + noll_val_fields + noll_idx_fields,
    start=t_start,
    end=t_end,
)

print(f"Total rows returned: {len(df_wfe)}")
if df_wfe.empty:
    print("No data — check NIGHT_DATE or the EFD alias.")
else:
    print(f"Time range: {df_wfe.index.min()}  →  {df_wfe.index.max()}")
    print(f"Unique sensorIds found: {sorted(df_wfe['sensorId'].unique())}")
    # Confirm Noll index mapping from first row
    print(f"\nNoll index mapping (first row): Z{int(df_wfe['nollZernikeIndices0'].iloc[0])} "
          f"to Z{int(df_wfe[f'nollZernikeIndices{N_NOLL_VALUES-1}'].iloc[0])}")
    print(f"Z4 (nollZernikeValues0) range: "
          f"{df_wfe['nollZernikeValues0'].min():.3f} to {df_wfe['nollZernikeValues0'].max():.3f} µm")
    print("\nFirst few rows:")
    display(df_wfe[["sensorId", "visitId", "nollZernikeValues0",
                    "nollZernikeValues1", "nollZernikeValues2"]].head(8))


Total rows returned: 623
Columns: ['sensorId', 'annularZernikeCoeff0', 'annularZernikeCoeff1', 'annularZernikeCoeff2', 'annularZernikeCoeff3', 'annularZernikeCoeff4', 'annularZernikeCoeff5', 'annularZernikeCoeff6', 'annularZernikeCoeff7', 'annularZernikeCoeff8', 'annularZernikeCoeff9', 'annularZernikeCoeff10', 'annularZernikeCoeff11', 'annularZernikeCoeff12', 'annularZernikeCoeff13', 'annularZernikeCoeff14', 'annularZernikeCoeff15', 'annularZernikeCoeff16', 'annularZernikeCoeff17', 'annularZernikeCoeff18']
Time range: 2025-11-16 00:29:23.175764+00:00  →  2025-11-16 07:18:54.912289+00:00

Unique sensorIds found: [np.int64(191), np.int64(195), np.int64(199), np.int64(203)]

First few rows:


,sensorId,annularZernikeCoeff0,annularZernikeCoeff1,annularZernikeCoeff2,annularZernikeCoeff3,annularZernikeCoeff4,annularZernikeCoeff5,annularZernikeCoeff6,annularZernikeCoeff7,annularZernikeCoeff8,annularZernikeCoeff9,annularZernikeCoeff10,annularZernikeCoeff11,annularZernikeCoeff12,annularZernikeCoeff13,annularZernikeCoeff14,annularZernikeCoeff15,annularZernikeCoeff16,annularZernikeCoeff17,annularZernikeCoeff18
2025-11-16 00:29:23.175764+00:00,191,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2025-11-16 00:29:23.280460+00:00,195,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2025-11-16 00:29:23.381750+00:00,199,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2025-11-16 00:29:23.482877+00:00,203,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2025-11-16 00:31:02.738430+00:00,191,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


### Sensor ID inspection

If the sensor IDs above differ from the defaults in `SENSOR_LABELS`, update the
mapping in the **Configuration** cell and re-run from there.

In [ ]:
# Per-sensor event count and Z4 (defocus) statistics
if not df_wfe.empty:
    z4_col = "nollZernikeValues0"  # Noll Z4 = defocus
    summary = (
        df_wfe.groupby("sensorId")[z4_col]
        .agg(N="count", mean="mean", std="std",
             min="min", median="median", max="max")
        .rename_axis("sensorId")
    )
    summary.insert(0, "label",
                   summary.index.map(lambda s: SENSOR_LABELS.get(int(s), f"sensor {s}")))
    print("Z4 (nollZernikeValues0, Noll defocus) statistics per sensor (µm):")
    display(summary.round(4))


Z4 (annularZernikeCoeff0) statistics per sensor (microns):


/sdf/group/rubin/sw/conda/envs/lsst-scipipe-12.1.0/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/sdf/group/rubin/sw/conda/envs/lsst-scipipe-12.1.0/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/sdf/group/rubin/sw/conda/envs/lsst-scipipe-12.1.0/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/sdf/group/rubin/sw/conda/envs/lsst-scipipe-12.1.0/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,label,N,mean,std,min,median,max
sensorId,,,,,,,
191,R00 (SW0),0,NaN,NaN,NaN,NaN,NaN
195,R04 (SW0),0,NaN,NaN,NaN,NaN,NaN
199,R40 (SW0),0,NaN,NaN,NaN,NaN,NaN
203,R44 (SW0),0,NaN,NaN,NaN,NaN,NaN


## Time Series: Z4 (Defocus) per Corner Sensor

In [ ]:
if df_wfe.empty:
    print("No data to plot.")
else:
    z4_col = "nollZernikeValues0"   # Noll Z4 defocus

    sensor_ids = sorted(df_wfe["sensorId"].unique())
    n_sensors  = len(sensor_ids)

    fig, axes = plt.subplots(
        n_sensors, 1,
        figsize=(13, 3.5 * n_sensors),
        sharex=True,
        squeeze=False,
    )

    for ax, sid in zip(axes[:, 0], sensor_ids):
        df_s  = df_wfe[df_wfe["sensorId"] == sid].sort_index()
        label = SENSOR_LABELS.get(int(sid), f"sensor {sid}")
        color = SENSOR_COLORS.get(int(sid), "gray")

        ax.plot(df_s.index, df_s[z4_col],
                marker="o", ms=4, lw=1.2, color=color, label=label)
        ax.axhline(0, color="black", lw=0.7, ls="--", alpha=0.5)

        median_val = df_s[z4_col].median()
        ax.axhline(median_val, color=color, lw=1.0, ls=":",
                   label=f"median = {median_val:.4f} µm")

        ax.set_ylabel("Z4 (µm)")
        ax.set_title(label, fontsize=10, loc="left")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.3)

    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[-1, 0].set_xlabel(f"UTC time on {NIGHT_DATE}")

    fig.suptitle(
        f"MTAOS Per-Sensor Z4 (Defocus) — {NIGHT_DATE}\n"
        f"nollZernikeValues0  [µm, Noll Z4]",
        fontsize=12,
    )
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


## All Four Sensors Overlaid

In [ ]:
if not df_wfe.empty:
    fig, ax = plt.subplots(figsize=(13, 4.5))

    for sid in sorted(df_wfe["sensorId"].unique()):
        df_s  = df_wfe[df_wfe["sensorId"] == sid].sort_index()
        label = SENSOR_LABELS.get(int(sid), f"sensor {sid}")
        color = SENSOR_COLORS.get(int(sid), "gray")
        ax.plot(df_s.index, df_s["nollZernikeValues0"],
                marker="o", ms=3, lw=1.0, alpha=0.8, color=color, label=label)

    ax.axhline(0, color="black", lw=0.7, ls="--", alpha=0.5)
    ax.set_ylabel("Z4 Defocus (µm)")
    ax.set_xlabel(f"UTC time on {NIGHT_DATE}")
    ax.set_title(f"MTAOS Z4 — all corner sensors — {NIGHT_DATE}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


## Bonus: Multiple Zernike Modes per Sensor

Grid of subplots — one row per Zernike mode, one column per sensor.

In [ ]:
# Which Noll indices to show (subset to keep the figure readable)
# Each Noll index N maps to nollZernikeValues{N - NOLL_OFFSET}
NOLL_MODES = [4, 5, 6, 7, 8, 11]   # defocus, astig, coma, spherical

NOLL_NAMES = {
    4: "Z4 defocus",
    5: "Z5 oblique astig",
    6: "Z6 vert astig",
    7: "Z7 vert coma",
    8: "Z8 horiz coma",
    9: "Z9", 10: "Z10",
    11: "Z11 spherical",
    12: "Z12", 13: "Z13",
}

if not df_wfe.empty:
    sensor_ids = sorted(df_wfe["sensorId"].unique())
    n_modes    = len(NOLL_MODES)
    n_sensors  = len(sensor_ids)

    fig, axes = plt.subplots(
        n_modes, n_sensors,
        figsize=(4.5 * n_sensors, 3.0 * n_modes),
        sharex="col", sharey="row",
        squeeze=False,
    )

    for col_idx, sid in enumerate(sensor_ids):
        df_s  = df_wfe[df_wfe["sensorId"] == sid].sort_index()
        label = SENSOR_LABELS.get(int(sid), f"sensor {sid}")
        color = SENSOR_COLORS.get(int(sid), "gray")

        for row_idx, noll in enumerate(NOLL_MODES):
            ax  = axes[row_idx][col_idx]
            col = f"nollZernikeValues{noll - NOLL_OFFSET}"

            if col not in df_s.columns:
                ax.set_visible(False)
                continue

            ax.plot(df_s.index, df_s[col],
                    marker="o", ms=2, lw=0.9, color=color)
            ax.axhline(0, color="black", lw=0.5, ls="--", alpha=0.4)
            ax.grid(True, alpha=0.25)

            if row_idx == 0:
                ax.set_title(label, fontsize=9)
            if col_idx == 0:
                ax.set_ylabel(f"{NOLL_NAMES.get(noll, f'Z{noll}')} (µm)", fontsize=8)
            if row_idx == n_modes - 1:
                ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
                ax.set_xlabel("UTC", fontsize=8)
                plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)

    fig.suptitle(
        f"MTAOS Annular Zernike Coefficients per Corner Sensor — {NIGHT_DATE}",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()
